# PDF로 RAG 만들고 FastAPI 서버에 연결하기 (9/29 퀴즈 대비)

- **1부**: 교재(기본편 CH04 LCEL · CH09~13) 순서대로 PDF RAG 체인을 만든다. 9/28 수업 `rag_02_0928.ipynb`와 같은 재료.
- **2부**: 만든 체인을 파일로 옮기고, FastAPI 서버의 `/ask` 주소에 연결한다. 퀴즈에서 "강사가 만든 서버에 RAG를 연결"하라는 부분이 이 모양일 가능성이 높다(강사 뼈대에 맞춰 이름만 바꾸면 된다).
- **3부**: RAGAS로 평가한다. 강사 데이터셋은 9/30에 받으므로, 지금은 임시 데이터셋(`ragas_testset_temp.csv`)으로 연습한다.
- 이 노트북은 `rag_two/01_class/week05/0928/` 폴더 안에 두고 실행한다(같은 폴더의 `data/virtual_tech_internal_handbook.pdf`를 읽는다).

## 0. 준비 — 설치와 API 키

처음 한 번만 설치 셀의 `#`을 지우고 실행한다. `.env` 파일에 `OPENAI_API_KEY`가 있어야 한다(값은 출력하지 않는다).

In [ ]:
# !pip install -qU langchain langchain-community langchain-openai faiss-cpu pymupdf
# !pip install -qU fastapi uvicorn requests

In [ ]:
from dotenv import load_dotenv

load_dotenv()   # .env의 OPENAI_API_KEY를 환경변수로 읽는다. 성공하면 True가 찍힌다

# 1부 — PDF RAG 체인 (교재 순서 8단계)

```
로드 → 분할 → 임베딩 → 벡터저장소 → 리트리버 → 프롬프트 → LLM → 체인
(CH09)  (CH10)  (CH11)    (CH12)      (CH13)     (CH05)   (CH07)  (CH04 LCEL)
```

## 1단계 · 로드 — PDF를 Document 리스트로 (CH09)

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

FILE_PATH = "./data/virtual_tech_internal_handbook.pdf"   # ① 읽을 PDF 경로

loader = PyMuPDFLoader(FILE_PATH)   # ② PDF 로더 객체를 만든다(아직 안 읽음)
docs = loader.load()                # ③ 실제로 읽는다. 페이지 1장 = Document 1개

print(len(docs))                    # ④ 페이지 수
print(docs[0].page_content[:200])   # ⑤ 첫 페이지 글자 200자

## 2단계 · 분할 — 긴 페이지를 조각(chunk)으로 (CH10)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # ① 조각 하나의 최대 글자 수
    chunk_overlap=100,   # ② 앞 조각과 겹치게 둘 글자 수(문장이 잘려 뜻이 끊기는 것을 줄임)
)
split_docs = text_splitter.split_documents(docs)   # ③ Document 리스트 → 더 잘게 자른 Document 리스트

print(len(split_docs))   # ④ 조각 수(페이지 수보다 많아진다)

## 3·4단계 · 임베딩과 벡터저장소 — 조각을 숫자 벡터로 바꿔 저장 (CH11·CH12)

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()                          # ① 글 → 숫자 벡터로 바꾸는 모델(OpenAI API 호출)
vectorstore = FAISS.from_documents(split_docs, embeddings)  # ② 조각마다 벡터를 만들어 FAISS에 저장

## 5단계 · 리트리버 — 질문과 가까운 조각을 찾아오는 객체 (CH13)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})   # ① 가까운 조각 4개를 돌려주는 리트리버

found = retriever.invoke("연차를 사용하려면 최소 며칠 전에 신청해야 하나요?")   # ② 검색만 해 본다
for i, d in enumerate(found, 1):
    print(i, d.metadata.get("page"), d.page_content[:60].replace("\n", " "))   # ③ 순위·쪽·앞부분

> 수업 파일(`rag_02_0928`)에는 `search_kwrgs`로 적혀 있다(철자 오타). 오타가 나도 에러는 안 나고 설정만 무시된다 — 기본값이 4라 결과가 같아 보였을 뿐이다. 정확한 이름은 `search_kwargs`.

## 6·7단계 · 프롬프트와 LLM (CH05·CH07)

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

prompt = PromptTemplate.from_template(
    """아래 [문서]만 근거로 질문에 한국어로 답하세요.
[문서]에 답이 없으면 "문서에서 찾을 수 없습니다"라고 답하세요.

[문서]
{context}

[질문]
{question}

[답변]"""
)   # ① {context}·{question} 두 빈칸이 있는 템플릿

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)   # ② 답을 만드는 모델. temperature=0 → 매번 비슷한 답

## 8단계 · 체인 — LCEL로 한 줄에 잇기 (CH04)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)   # ① 찾은 조각 여러 개 → 문자열 하나

rag_chain = (
    {"context": retriever | format_docs,       # ② 질문 → 리트리버 → 문자열로 합쳐 {context}에
     "question": RunnablePassthrough()}        # ③ 질문은 그대로 {question}에 (CH04-10)
    | prompt                                    # ④ 두 빈칸을 채운 프롬프트
    | llm                                       # ⑤ 모델이 답 생성
    | StrOutputParser()                         # ⑥ 답 객체 → 글자만 꺼냄
)

In [ ]:
print(rag_chain.invoke("연차를 사용하려면 최소 며칠 전에 신청해야 하나요?"))

- 실행 순서: `invoke(질문)` → ②와 ③이 **동시에** 돈다(딕셔너리 = RunnableParallel, CH04-11) → ④ → ⑤ → ⑥ → 글자 답 1개가 출력된다.
- `format_docs`는 일반 함수지만 `|` 안에 들어가면 LangChain이 RunnableLambda로 감싼다(CH04-12).

# 2부 — 서버에 연결하기

서버(FastAPI)는 노트북 셀 안에서 켜 두면 셀이 끝나지 않고 멈춘다. 그래서 실무처럼 **파일 두 개**로 나눈다.

| 파일 | 하는 일 |
|---|---|
| `rag_core.py` | 1부 코드를 함수 하나(`build_rag_chain`)로 묶은 것 |
| `server.py` | FastAPI 서버. 켜질 때 `build_rag_chain()`을 한 번 불러 체인을 만들고, `/ask`로 온 질문을 체인에 넘긴다 ← 이게 "연결" |

아래 두 셀의 `%%writefile`은 셀 내용을 그 이름의 파일로 저장한다(실행은 하지 않는다).

In [ ]:
%%writefile rag_core.py
from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

PROMPT = PromptTemplate.from_template(
    """아래 [문서]만 근거로 질문에 한국어로 답하세요.
[문서]에 답이 없으면 "문서에서 찾을 수 없습니다"라고 답하세요.

[문서]
{context}

[질문]
{question}

[답변]"""
)


def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


def build_rag_chain(file_path="./data/virtual_tech_internal_handbook.pdf"):
    """PDF 경로를 받아 1부와 같은 RAG 체인을 만들어 돌려준다."""
    docs = PyMuPDFLoader(file_path).load()                                   # 1 로드
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    split_docs = splitter.split_documents(docs)                              # 2 분할
    vectorstore = FAISS.from_documents(split_docs, OpenAIEmbeddings())       # 3·4 임베딩·저장
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})             # 5 리트리버
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)                     # 7 LLM
    return (                                                                 # 8 체인
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | PROMPT
        | llm
        | StrOutputParser()
    )

In [ ]:
%%writefile server.py
from fastapi import FastAPI
from pydantic import BaseModel
from rag_core import build_rag_chain    # ① rag_core.py의 함수를 가져온다

app = FastAPI()                         # ② 서버 객체. uvicorn server:app 의 app이 이것
rag_chain = build_rag_chain()           # ③ 서버가 켜질 때 한 번만 체인을 만든다(PDF 읽기·임베딩)


class Question(BaseModel):              # ④ 요청 본문 모양: {"question": "..."}
    question: str


@app.post("/ask")                       # ⑤ POST /ask 로 요청이 오면 아래 함수 실행
def ask(q: Question):
    answer = rag_chain.invoke(q.question)   # ⑥ 연결: 받은 질문을 RAG 체인에 넘긴다
    return {"question": q.question, "answer": answer}   # ⑦ JSON으로 돌려준다

## 서버 켜기 — 터미널에서 (노트북 셀 아님)

이 노트북이 있는 폴더(`0928`)에서 가상환경을 켠 뒤:

Windows (수업)
```
.venv\Scripts\activate
uvicorn server:app --reload
```

Mac (집)
```
source .venv/bin/activate
uvicorn server:app --reload
```

- `server:app` = `server.py` 파일 안의 `app` 변수.
- `Uvicorn running on http://127.0.0.1:8000`이 보이면 켜진 것. 터미널은 켜 둔 채로 둔다(닫으면 서버도 꺼진다).
- 브라우저에서 `http://127.0.0.1:8000/docs`를 열면 `/ask`를 직접 눌러 시험할 수 있다(FastAPI 자동 문서).
- 가상환경 폴더가 한 단계 위(`rag_two/.venv`)에 있으면 경로만 맞춘다.

## 노트북에서 서버에 질문 보내기

In [ ]:
import requests

res = requests.post(
    "http://127.0.0.1:8000/ask",                                   # ① 켜 둔 서버의 /ask 주소
    json={"question": "서울 출장 시 숙박비는 최대 얼마까지 지원되나요?"},   # ② Question 모양에 맞춘 본문
)
print(res.status_code)   # ③ 200이면 성공
print(res.json())        # ④ {"question": ..., "answer": ...}

- `ConnectionError`가 나면 서버가 안 켜진 것이다(터미널 확인).
- `422`가 나면 본문 모양이 틀린 것이다(`"question"` 철자).
- `500`이 나면 서버 안 체인에서 에러가 난 것이다 — 터미널에 찍힌 에러를 본다(대부분 API 키 또는 PDF 경로).

# 3부 — RAGAS로 평가하기 (교재 Advanced A-CH05 §2)

**데이터셋은 언제·어디에 넣나**

| 언제 | 무엇을 | 어디에 |
|---|---|---|
| 지금(연습) | 임시 데이터셋 `ragas_testset_temp.csv`(질문 5개 + 정답) | 이 노트북 옆 `data/` 폴더 |
| 9/30 퀴즈 | 강사가 준 데이터셋 파일 | 같은 `data/` 폴더에 넣고, 아래 `DATASET_PATH` **한 줄만** 바꾼다 |

**평가가 하는 일**: 데이터셋의 질문마다 ① 내 RAG가 찾아온 문서(contexts)와 ② 내 RAG의 답(response)을 모은 뒤, 정답(reference)과 함께 채점 모델(LLM)에게 넘겨 0~1 점수를 받는다.

```
데이터셋(질문·정답) → 질문마다 내 RAG 실행(문서·답 모으기) → EvaluationDataset → evaluate() → 점수표
```

In [ ]:
# 처음 한 번만. ragas 최신판(0.4.3)은 langchain-community 0.4와 함께 설치하면 import 에러가 난다(9/29 집에서 확인)
# → langchain-community를 0.4 미만으로 같이 설치한다
# !pip install -qU ragas "langchain-community<0.4" pandas

In [ ]:
import pandas as pd

DATASET_PATH = "./data/ragas_testset_temp.csv"   # ① 퀴즈 때는 이 경로만 강사 파일로 바꾼다

df = pd.read_csv(DATASET_PATH)   # ② CSV를 표(DataFrame)로 읽는다
print(df.columns.tolist())       # ③ 칸 이름부터 확인한다 — 아래 ④에서 맞춰야 한다
df.head()

In [ ]:
# ④ 칸 이름 맞추기. RAGAS가 쓰는 이름은 user_input(질문), reference(정답).
#    강사 파일 칸 이름이 다르면(교재 옛 이름 question·ground_truth 등) 여기서 바꾼다. 이미 같으면 아무 일도 안 한다
df = df.rename(columns={"question": "user_input", "ground_truth": "reference"})
print(df.columns.tolist())

In [ ]:
# ⑤ 질문마다 내 RAG를 돌려 '찾아온 문서'와 '답'을 모은다 — 1부의 retriever, rag_chain을 그대로 쓴다
rows = []
for question, reference in zip(df["user_input"], df["reference"]):
    contexts = [d.page_content for d in retriever.invoke(question)]   # 찾아온 문서 글자 리스트
    answer = rag_chain.invoke(question)                                # 내 RAG의 답
    rows.append({
        "user_input": question,          # 질문
        "retrieved_contexts": contexts,  # 내 RAG가 찾아온 문서들
        "response": answer,              # 내 RAG의 답
        "reference": reference,          # 정답(데이터셋)
    })

print(len(rows), "문항 준비")
print(rows[0]["response"])   # 첫 질문의 답만 확인

> 퀴즈에서 "서버를 거쳐" 평가하라고 하면 `answer = rag_chain.invoke(question)` 한 줄을
> `answer = requests.post("http://127.0.0.1:8000/ask", json={"question": question}).json()["answer"]`로 바꾼다(2부 서버가 켜져 있어야 함).

In [ ]:
from ragas import EvaluationDataset, evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

dataset = EvaluationDataset.from_list(rows)   # ⑥ 모은 rows를 RAGAS용 데이터셋으로

result = evaluate(                            # ⑦ 채점. 문항 × 지표 수만큼 채점 모델을 부른다(1~2분)
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0)),   # 채점하는 모델
    embeddings=LangchainEmbeddingsWrapper(OpenAIEmbeddings()),                 # answer_relevancy에 필요
)
result   # ⑧ 지표별 평균 점수가 한 줄로 나온다

In [ ]:
result.to_pandas()   # ⑨ 문항별 점수표. 점수가 낮은 줄을 보고 어디서(검색? 답?) 틀렸는지 찾는다

**지표 4개 — 무엇을 재나** (모두 0~1, 높을수록 좋다)

| 지표 | 묻는 것 | 낮으면 어디를 고치나 |
|---|---|---|
| faithfulness(충실도) | 답이 찾아온 문서에 근거했나(지어낸 말이 없나) | 프롬프트("문서만 근거로") |
| answer_relevancy(답변 관련성) | 답이 질문에 맞는 말인가 | 프롬프트·모델 |
| context_precision(문맥 정밀도) | 쓸모 있는 문서가 위 순위에 있나 | 리트리버(k, 리랭커) |
| context_recall(문맥 재현율) | 정답에 필요한 내용을 다 찾아왔나 | 분할(chunk_size)·리트리버 |

- 앞의 둘은 **답** 쪽, 뒤의 둘은 **검색** 쪽 점수다. 점수표를 보고 "검색이 문제인지, 답 만들기가 문제인지"를 가르는 것이 평가의 목적.
- `NaN`이 보이면 그 칸 채점이 실패한 것이다(API 키·요금·시간 초과). 다시 실행하면 대부분 채워진다.